# Juliet stratified cohort + Router suite 학습

Frozen split을 변경하지 않고 Train 6,000 / Dev 1,500을 Expert→CWE→leakage-group 다양성 우선으로 선택합니다. E6는 전량 보존합니다. 각 case의 후보와 5개 Expert 작업은 하나의 물리 API 요청으로 처리되며 case별로 체크포인트됩니다.

In [ ]:
from pathlib import Path

EVAL_ROOT = Path.cwd().resolve()
if EVAL_ROOT.name != 'Model_Evaluation':
    EVAL_ROOT = (EVAL_ROOT / 'Model_Evaluation').resolve()
CONFIG_PATH = EVAL_ROOT / 'configs' / 'full.toml'
COHORT_CONFIG_PATH = EVAL_ROOT / 'configs' / 'cohort_15837.toml'
ENV_FILE = EVAL_ROOT.parent / '.env'
MAX_CANDIDATES_PER_CASE = 4
HARD_NEGATIVES_PER_CASE = 1
MAX_CONCURRENCY = 1000  # OpenRouter 한도에 맞춰 낮출 수 있음
TARGET_TRUTH_RECALL = 0.95
RUN_LEARNING_CURVES = True


In [ ]:
import json, sys
sys.path.insert(0, str(EVAL_ROOT / 'src'))
from model_evaluation.config import load_config, load_mapping
from model_evaluation.stages.select_cohort import load_cohort_config, ensure_frozen_index, build_cohort_manifests
from model_evaluation.stages.materialize_dataset import materialize_dataset
from model_evaluation.candidates import cache_candidates
from model_evaluation.workflow import (resolve_models, plan_outcome_matrix, collect_outcome_matrix, audit_outcome_matrix, train_utility_router, train_router_learning_curves)
from model_evaluation.adapters.llm_security import expert_assignments

config = load_config(CONFIG_PATH)
mapping = load_mapping(config.paths.mapping)
cohort_config = load_cohort_config(COHORT_CONFIG_PATH)
models = resolve_models(ENV_FILE)
COHORT_DIR = EVAL_ROOT / 'work' / 'cohort_15837'
RUN_DIR = EVAL_ROOT / 'work' / 'router_training_stratified_7500'
RESULT_DIR = EVAL_ROOT / 'results' / 'router_training_stratified_7500'
ARTIFACT = EVAL_ROOT / 'artifacts' / 'juliet_utility_router.pkl'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print('Physical model:', models[0])

## 1. Leakage-safe stratified cohort manifest

In [ ]:
index_report = ensure_frozen_index(config, mapping, progress=print)
cohort_report = build_cohort_manifests(config, cohort_config, output_directory=COHORT_DIR)
print(json.dumps(index_report, ensure_ascii=False, indent=2))
print(json.dumps(cohort_report, ensure_ascii=False, indent=2))

## 2. Train/Dev materialization과 Semantic Analyzer cache

In [ ]:
manifests = {split: COHORT_DIR / f'cohort_{split}.jsonl' for split in ('train', 'dev')}
materialization = materialize_dataset(
    config, mapping, output_directory=RUN_DIR / 'cases', splits=('train', 'dev'),
    selection_manifests=manifests, progress=print,
)
candidate_reports = {}
for split in ('train', 'dev'):
    candidate_reports[split] = cache_candidates(
        RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
        max_source_bytes=config.max_source_bytes, parse_timeout_ms=config.parse_timeout_ms, progress=print,
    )
print(json.dumps({'materialization': materialization, 'candidates': candidate_reports}, ensure_ascii=False, indent=2))

## 3. API 호출 계획

In [ ]:
plans = {}
for split in ('train', 'dev'):
    plans[split] = plan_outcome_matrix(
        cases_path=RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        candidate_cache=RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
        selection_manifest=RUN_DIR / 'selections' / f'selected_{split}.jsonl',
        outcome_path=RUN_DIR / 'outcomes' / f'outcomes_{split}.jsonl',
        model_ids=models, max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
        hard_negatives_per_case=HARD_NEGATIVES_PER_CASE,
    )
print(json.dumps(plans, ensure_ascii=False, indent=2))

## 4. Batched Expert outcome 수집 (case당 API 최대 1회, 최대 1,000건 비동기 동시 처리)

In [ ]:
collection_reports = {}
for split in ('train', 'dev'):
    report = collect_outcome_matrix(
        env_file=ENV_FILE, cases_path=RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        candidate_cache=RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
        outcome_path=RUN_DIR / 'outcomes' / f'outcomes_{split}.jsonl',
        ledger_path=RUN_DIR / 'ledgers' / f'{split}_api_ledger.jsonl',
        model_ids=models, max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
        hard_negatives_per_case=HARD_NEGATIVES_PER_CASE, max_concurrency=MAX_CONCURRENCY,
    )
    collection_reports[split] = report
    print(split, json.dumps(report, ensure_ascii=False, indent=2))
    if report['status'] != 'complete':
        print('실패한 case만 남았습니다. 성공한 case는 저장되었으며 이 셀을 다시 실행하면 실패 case부터 재개합니다.')
        break

## 5. LR vs GBDT vs Multi-task MLP + 기존 Escalation Gate

In [ ]:
expected_ids = [item.assignment_id for item in expert_assignments(models)]
outcome_files = {split: RUN_DIR / 'outcomes' / f'outcomes_{split}.jsonl' for split in ('train', 'dev')}
audits = {
    split: audit_outcome_matrix(
        path, expected_assignment_ids=expected_ids,
        selection_manifest=RUN_DIR / 'selections' / f'selected_{split}.jsonl',
    ) if path.exists() else {'complete': False, 'reason': 'missing'}
    for split, path in outcome_files.items()
}
print(json.dumps(audits, ensure_ascii=False, indent=2))
if all(item['complete'] for item in audits.values()):
    training_report = train_utility_router(
        train_outcomes=outcome_files['train'], dev_outcomes=outcome_files['dev'],
        train_selection_manifest=RUN_DIR / 'selections' / 'selected_train.jsonl',
        dev_selection_manifest=RUN_DIR / 'selections' / 'selected_dev.jsonl',
        train_cohort_manifest=manifests['train'],
        artifact_path=ARTIFACT, report_path=RESULT_DIR / 'training_report.json',
        model_ids=models, seed=config.seed, target_truth_recall=TARGET_TRUTH_RECALL,
    )
    print(json.dumps(training_report, ensure_ascii=False, indent=2))
    if RUN_LEARNING_CURVES:
        curve_report = train_router_learning_curves(
            train_outcomes=outcome_files['train'], dev_outcomes=outcome_files['dev'],
            train_cohort_manifest=manifests['train'],
            report_path=RESULT_DIR / 'learning_curves.json', seed=config.seed,
            target_truth_recall=TARGET_TRUTH_RECALL,
        )
        print(json.dumps(curve_report, ensure_ascii=False, indent=2))
else:
    print('Outcome matrix가 아직 완성되지 않았습니다. 4번 셀을 다시 실행해 남은 case를 수집하세요.')